# Manga OCR with learned speaker matching

Submission notebook: training, all 42 speaker configurations, final refit and test inference.
Set DATA_ROOT in the setup cell and use a CUDA-enabled Python kernel. Outputs go to new_run; feature caches go to feature_cache. The pretrained revision is pinned.
The selected validation balanced joint F1 is 0.4268. This notebook has no saved outputs; results are included separately.


## Install missing notebook dependencies
Run once in `ncl_sae`, then restart the kernel. This changes packages in that environment,
including pinning transformers and scikit-learn for this notebook, but keeps its GPU
PyTorch/torchvision installation. If you need to preserve other projects' dependencies,
clone the environment first. Subsequent runs can skip this cell.

In [ ]:
%pip install -q transformers==4.44.2 timm==1.0.9 einops==0.8.0 pulp==2.9.0 sentencepiece==0.2.0 rapidfuzz==3.9.7 scikit-learn==1.5.2 joblib==1.4.2
%pip install -q "numpy<2" pandas scipy pillow tqdm ipywidgets nbformat

In [ ]:
from pathlib import Path
import json, re, random, hashlib, time, sys, importlib.util, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm
from rapidfuzz.fuzz import ratio
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path("dataset").resolve()  # Set to the supplied dataset directory.
WORK = Path('new_run').resolve()
PREVIOUS_RUN = Path('feature_cache').resolve()
WORK.mkdir(parents=True, exist_ok=True)
RUN_STARTED = time.perf_counter()
TARGET_SECONDS = 2 * 60 * 60
USE_AMP = True  # BF16 reduces activation memory; set False if it causes numerical errors.
OCR_BATCH = 8
EMBED_BATCH = 8
print('Python:', sys.executable)
if not torch.cuda.is_available():
    raise RuntimeError('This kernel cannot use CUDA. Select ncl_sae, not the system Python.')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
assert not USE_AMP or torch.cuda.is_bf16_supported(), 'Set USE_AMP=False on this GPU.'
probe = torch.randn(128, 128, device='cuda')
print('CUDA check:', float((probe @ probe).mean()))
del probe

manifest = json.loads((DATA_ROOT / 'sequences.json').read_text())
seq_by_id = {s['sequence_id']: s for s in manifest}
assert len(seq_by_id) == len(manifest), 'Duplicate sequence IDs'
for s in manifest:
    assert len(s['images']) == 3
    for p in s['images']:
        assert (DATA_ROOT / p).is_file(), p

spec = importlib.util.spec_from_file_location('task_score', DATA_ROOT / 'score.py')
scorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer)
gold = scorer.load_jsonl(DATA_ROOT / 'development/labels.jsonl', reference=True)
dev_ids = sorted(s['sequence_id'] for s in manifest if s['split'] == 'development')
test_ids = [s['sequence_id'] for s in manifest if s['split'] == 'test']
assert set(gold) == set(dev_ids)
assert len(dev_ids) == 80 and len(test_ids) == 15
assert torch.cuda.is_available(), 'CUDA is required for this run.'
print(DATA_ROOT, torch.cuda.get_device_name(0))
print(f'{len(dev_ids)} development sequences; {len(test_ids)} test sequences')
# Load an untouched scorer for final verification of every selected metric.
official_spec = importlib.util.spec_from_file_location('official_score', DATA_ROOT / 'score.py')
official_score = importlib.util.module_from_spec(official_spec)
official_spec.loader.exec_module(official_score)


## Split whole sequences
Exact duplicate pages are grouped so that they cannot cross the split. This does not
detect neighboring pages from the same book. If you know the book/chapter groups,
populate `BOOK_GROUPS` before running this cell for a stronger holdout.

In [ ]:
BOOK_GROUPS = {}  # Optional: sequence_id -> book/chapter group.
parent = {sid: sid for sid in dev_ids}
def root(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def unite(a, b):
    parent[root(b)] = root(a)
seen = {}
for sid in dev_ids:
    for rel in seq_by_id[sid]['images']:
        digest = hashlib.sha256((DATA_ROOT / rel).read_bytes()).hexdigest()
        if digest in seen:
            unite(sid, seen[digest])
        seen[digest] = sid
book_seen = {}
for sid, book in BOOK_GROUPS.items():
    assert sid in parent
    if book in book_seen:
        unite(sid, book_seen[book])
    book_seen[book] = sid
groups = [root(sid) for sid in dev_ids]
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
              .split(dev_ids, groups=groups))
train_ids = [dev_ids[i] for i in tr]
val_ids = [dev_ids[i] for i in va]
assert not {groups[i] for i in tr} & {groups[i] for i in va}
(WORK / 'split.json').write_text(json.dumps({'train': train_ids, 'validation': val_ids}, indent=2))
print('Train:', len(train_ids), 'Validation:', len(val_ids))

## Frozen detector and OCR
Magi already orders its text boxes using panels. Keep that order. An empty character
bank makes its naming API return `Other`, so we use crop embeddings and local cluster
IDs instead. No real names or reference speaker labels enter inference.

The resolved model revision is saved. Keep `run_metadata.json` to reproduce the run;
put its revision in `MODEL_REVISION` when rerunning elsewhere. Loading this model
executes its custom Python code (`trust_remote_code=True`).

In [ ]:
from huggingface_hub import model_info
from transformers import AutoModel

MODEL_ID = 'ragavsachdeva/magiv2'
MODEL_REVISION = "fbc890fec52977142e8ee00bfe26e9458b65517c"  # Use the saved commit SHA to reproduce an earlier run.
prior = WORK / 'run_metadata.json'
prior = prior if prior.exists() else PREVIOUS_RUN / 'run_metadata.json'
saved_revision = json.loads(prior.read_text()).get('revision') if prior.exists() else None
revision = MODEL_REVISION or saved_revision or model_info(MODEL_ID).sha
model = AutoModel.from_pretrained(MODEL_ID, revision=revision,
                                  trust_remote_code=True).cuda().eval()
metadata = {'model': MODEL_ID, 'revision': revision, 'seed': SEED,
            'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
            'extractor_version': 2, 'ocr_max_new_tokens': 128,
            'amp': USE_AMP, 'ocr_batch': OCR_BATCH, 'embed_batch': EMBED_BATCH}
(WORK / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
(WORK / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
CACHE = PREVIOUS_RUN / ('cache_' + revision[:12])
CACHE.mkdir(parents=True, exist_ok=True)

def read_image(rel):
    with Image.open(DATA_ROOT / rel) as im:
        return np.array(im.convert('L').convert('RGB'))

def extract_sequence(sid):
    paths = seq_by_id[sid]['images']
    fingerprint = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode())
    for rel in paths:
        fingerprint.update((DATA_ROOT / rel).read_bytes())
    signature = fingerprint.hexdigest()
    destination = CACHE / f'{sid}.joblib'
    if destination.exists():
        cached = joblib.load(destination)
        if cached['signature'] == signature:
            return cached['pages']
    pages = []
    torch.cuda.synchronize()
    started = time.time()
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16, enabled=USE_AMP):
        for rel in paths:
            image = read_image(rel)
            result = model.predict_detections_and_associations([image])[0]
            boxes = result['texts']
            texts = model.predict_ocr([image], [boxes], batch_size=OCR_BATCH,
                                      max_new_tokens=128)[0]
            # Visual text features let the small filter learn beyond word matching.
            text_emb = model.predict_crop_embeddings([image], [boxes], batch_size=EMBED_BATCH)[0]
            char_emb = model.predict_crop_embeddings(
                [image], [result['characters']], batch_size=EMBED_BATCH)[0]
            assert len(texts) == len(boxes) == len(result['is_essential_text'])
            result.update(ocr=texts, size=[image.shape[1], image.shape[0]],
                          text_embeddings=text_emb.float().cpu().numpy(),
                          char_embeddings=char_emb.float().cpu().numpy())
            pages.append(result)
    torch.cuda.synchronize()
    tmp = destination.with_suffix('.tmp')
    joblib.dump({'signature': signature, 'pages': pages}, tmp)
    tmp.replace(destination)
    with (WORK / 'extraction_log.jsonl').open('a') as f:
        f.write(json.dumps({'sequence_id': sid, 'seconds': time.time()-started,
                            'regions': sum(len(p['ocr']) for p in pages)}) + '\n')
    return pages

# Time three training sequences; the first includes CUDA warm-up.
smoke_ids = train_ids[:3]
smoke_seconds = []
for smoke_id in smoke_ids:
    t0 = time.perf_counter()
    smoke = extract_sequence(smoke_id)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    smoke_seconds.append(seconds)
    print(f'{smoke_id}: {seconds:.1f}s; regions: {[len(p["ocr"]) for p in smoke]}')
display(pd.DataFrame({'text': smoke[0]['ocr'], 'essential': smoke[0]['is_essential_text']}))

logs = []
for log_path in [PREVIOUS_RUN / 'extraction_log.jsonl', WORK / 'extraction_log.jsonl']:
    if log_path.exists():
        logs.extend(json.loads(line) for line in log_path.read_text().splitlines() if line.strip())
recent = {r['sequence_id']: r['seconds'] for r in logs}
sample = [recent[sid] for sid in smoke_ids]
remaining = sum(not (CACHE / f'{sid}.joblib').exists() for sid in dev_ids + test_ids)
elapsed = time.perf_counter() - RUN_STARTED
estimate = elapsed + float(np.median(sample)) * remaining + 15 * 60
conservative = elapsed + max(sample) * remaining * 1.3 + 20 * 60
print(f'Rough total: {estimate/60:.0f} min; conservative estimate: {conservative/60:.0f} min')
print('Includes elapsed setup plus a scoring allowance; page complexity and download speeds vary.')
if conservative > TARGET_SECONDS:
    print('WARNING: the two-hour target is not supported by this timing. No pages will be skipped.')
(WORK / 'timing_estimate.json').write_text(json.dumps({
    'sample_seconds': sample, 'remaining_sequences': remaining,
    'estimated_total_minutes': estimate/60, 'conservative_minutes': conservative/60}, indent=2))


If the smoke test fails, fix the environment before running the full cache. A crash
does not create a fake empty prediction. Completed sequences can be reused on rerun.

In [ ]:
dev_cache = {sid: extract_sequence(sid) for sid in tqdm(dev_ids, desc='Development OCR')}

## Weak labels from the training transcripts
There are no balloon boxes in the references. Align OCR regions to short spans of
reference text. Confident matches teach inclusion and narration. Very dissimilar,
nonessential regions on pages with good reference coverage provide weak negatives.
Everything ambiguous is ignored. These negatives can still be wrong; the audit table
below is part of the experiment, not ground truth.

This first version does not solve all split/merged-balloon cases. Short punctuation
is accepted only on an exact match. Validation labels are never used to fit either head.

In [ ]:
def clean(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

def reference_candidates(rows):
    candidates = []
    for start in range(len(rows)):
        for width in range(1, 4):
            chunk = rows[start:start+width]
            if len(chunk) != width:
                continue
            speakers = {r['speaker'] for r in chunk}
            if width > 1 and len(speakers) != 1:
                continue
            candidates.append((scorer.norm(' '.join(r['text'] for r in chunk)),
                               chunk[0]['speaker'], set(range(start, start+width))))
    return candidates

def match_page(page, rows):
    candidates = reference_candidates(rows)
    matches = []
    covered = set()
    for text in page['ocr']:
        query = scorer.norm(text)
        ranked = sorted([(ratio(query, target)/100, speaker, indices, target)
                         for target, speaker, indices in candidates],
                        key=lambda x: x[0], reverse=True)
        best = ranked[0] if ranked else (0., None, set(), '')
        confidence = best[0] >= 0.90 and (len(query) >= 4 or query == best[3])
        # Repeated wording with different identities is unsafe narration supervision.
        ambiguous = any(r[1] != best[1] and r[0] >= best[0]-0.03 for r in ranked[1:])
        matches.append((best, confidence, ambiguous))
        if confidence:
            covered |= best[2]
    return matches, len(covered)/max(1, len(rows)) if rows else 1.

def region_features(page, i):
    w, h = page['size']
    x1, y1, x2, y2 = page['texts'][i]
    text = clean(page['ocr'][i])
    n = max(len(text), 1)
    associated = {int(t) for t, c in page['text_character_associations']}
    tails = {int(t) for t, c in page.get('text_tail_associations', [])}
    geometry = [x1/w, y1/h, x2/w, y2/h, (x2-x1)/w, (y2-y1)/h,
                np.log1p(len(text)), len(text.split())/50,
                sum(c.isalpha() for c in text)/n,
                sum(c.isdigit() for c in text)/n,
                sum(c in '.!?' for c in text)/n,
                float(page['is_essential_text'][i]), float(i in associated), float(i in tails)]
    emb = np.asarray(page['text_embeddings'][i], dtype=np.float32)
    emb = emb / max(float(np.linalg.norm(emb)), 1e-8)
    return np.concatenate([geometry, emb]).astype(np.float32)

def training_examples(ids):
    X, y, nx, ny, audit = [], [], [], [], []
    for sid in ids:
        for p, (page, rows) in enumerate(zip(dev_cache[sid], gold[sid])):
            matches, coverage = match_page(page, rows)
            for i, (best, confident, ambiguous) in enumerate(matches):
                if not clean(page['ocr'][i]):
                    continue
                label = None
                if confident:
                    label = 1
                elif best[0] < 0.25 and coverage >= 0.8 and not page['is_essential_text'][i]:
                    label = 0
                if label is not None:
                    features = region_features(page, i)
                    X.append(features); y.append(label)
                    audit.append({'sequence_id': sid, 'page': p+1, 'region': i,
                                  'ocr': page['ocr'][i], 'weak_include': label,
                                  'similarity': best[0], 'reference_coverage': coverage})
                    if confident and not ambiguous and best[1] is not None:
                        nx.append(features); ny.append(int(best[1] == 'NARRATION'))
    return np.asarray(X), np.asarray(y), np.asarray(nx), np.asarray(ny), pd.DataFrame(audit)

def fit_heads(ids):
    X, y, nx, ny, audit = training_examples(ids)
    if len(set(y)) < 2 or len(set(ny)) < 2:
        raise RuntimeError('Insufficient weak classes. Inspect training alignment; do not fabricate labels.')
    def fit(x, target):
        learner = make_pipeline(StandardScaler(), LogisticRegression(
            C=0.05, class_weight='balanced', max_iter=2000, random_state=SEED))
        return learner.fit(x, target)
    return {'story': fit(X, y), 'narration': fit(nx, ny)}, audit, {
        'story_counts': dict(Counter(map(int, y))), 'narration_counts': dict(Counter(map(int, ny)))}

heads, audit, counts = fit_heads(train_ids)
audit.to_csv(WORK / 'training_alignment_audit.csv', index=False)
(WORK / 'training_counts.json').write_text(json.dumps(counts, indent=2))
joblib.dump(heads, WORK / 'validation_heads.joblib')
print(counts)
display(audit.sample(min(15, len(audit)), random_state=SEED))

## Trainable speaker components
We start with confident OCR matches to known reference speakers. Conflicting local
clusters are left unlabelled. The matcher learns from cross-page cluster pairs; it uses
five similarity features after a training-only PCA projection. The ranker learns from
geometry, panel overlap and tail-box proximity. It does not receive the upstream
speaker choice as a feature.

A stricter first experiment required two agreeing anchors and used many pair features.
It produced only 76 pairs and scored 0.3525 balanced joint F1 with both components.
The simpler version below admits one unambiguous anchor and uses fewer features.
It produced 442 pairs and 273 ranker-training balloons in the recorded run. This adds
coverage at the cost of noisier targets. The current balloon is still excluded from
its own ranker target.

Character merges keep the existing constraints: no two distinct local clusters from
one page may merge, and every cross-group pair must pass the threshold. The ranker
only overrides a link when its best cluster has both a high score and a clear margin.
Otherwise it keeps the frozen association. Off-panel speech remains an open problem.

In [ ]:
"""Small speaker models trained from conservative development-only anchors."""
import numpy as np
from collections import defaultdict, Counter
from rapidfuzz.fuzz import ratio
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def normalize_vector(x):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), 1e-8)

def page_clusters(page):
    groups = defaultdict(list)
    for c, label in enumerate(page['character_cluster_labels']):
        groups[int(label)].append(c)
    return groups

def transcript_anchors(page, rows, normalizer):
    # A repeated phrase with competing speakers is not a reliable identity anchor.
    anchors = {}
    for i, raw in enumerate(page['ocr']):
        text = normalizer(raw)
        if len(text) < 8:
            continue
        scores = sorted([(ratio(text, normalizer(r['text']))/100, r['speaker'])
                         for r in rows], key=lambda x: x[0], reverse=True)
        if not scores or scores[0][0] < 0.94:
            continue
        best, label = scores[0]
        if label in (None, 'UNKNOWN', 'NARRATION'):
            continue
        if any(speaker != label and score >= best-0.05 for score, speaker in scores[1:]):
            continue
        anchors[i] = label
    return anchors

def anchor_votes(page, anchors):
    votes = defaultdict(list)
    for t, c in page['text_character_associations']:
        if t in anchors:
            votes[int(page['character_cluster_labels'][c])].append((t, anchors[t]))
    return votes

def agreed_identity(votes, exclude=None, minimum=1):
    labels = [label for text, label in votes if text != exclude]
    # The ranker excludes the current balloon. Contradictory anchors are ignored.
    return labels[0] if len(labels) >= minimum and len(set(labels)) == 1 else None

def cluster_nodes(pages):
    nodes, appearances = [], {}
    for p, page in enumerate(pages):
        for label, members in page_clusters(page).items():
            prototype = normalize_vector(np.mean([
                normalize_vector(page['char_embeddings'][c]) for c in members], axis=0))
            index = len(nodes)
            nodes.append({'page': p, 'local': label, 'embedding': prototype})
            for c in members:
                appearances[p, c] = index
    return nodes, appearances

def panel_for(box, panels):
    x1, y1, x2, y2 = box
    area = max((x2-x1)*(y2-y1), 1.)
    overlap = [max(0., min(x2,b[2])-max(x1,b[0])) * max(0., min(y2,b[3])-max(y1,b[1])) / area for b in panels]
    return int(np.argmax(overlap)) if overlap and max(overlap) >= 0.3 else -1

def box_info(box, size):
    w, h = size
    x1, y1, x2, y2 = np.asarray(box) / np.array([w,h,w,h])
    return np.array([(x1+x2)/2, (y1+y2)/2, max(x2-x1,0), max(y2-y1,0)])

def link_features(page, text, char):
    a = box_info(page['texts'][text], page['size'])
    b = box_info(page['characters'][char], page['size'])
    delta = b[:2]-a[:2]
    tp, cp = panel_for(page['texts'][text],page['panels']), panel_for(page['characters'][char],page['panels'])
    links = {int(t): int(c) for t,c in page['text_character_associations']}
    old = links.get(text)
    labels = page['character_cluster_labels']
    tails = {int(t):int(tail) for t,tail in page.get('text_tail_associations',[])}
    tail = tails.get(text)
    td = box_info(page['tails'][tail],page['size'])[:2]-b[:2] if tail is not None else np.zeros(2)
    same_old_cluster = old is not None and labels[old] == labels[char]
    # Tail boxes provide proximity, not an inferred tail direction.
    return np.r_[a,b,delta,np.abs(delta),np.linalg.norm(delta),
                 tp >= 0 and tp == cp, tp < 0 or cp < 0,
                 tail is not None,
                 td,np.linalg.norm(td),
                 sum(label == labels[char] for label in labels)/max(len(labels),1)].astype(np.float32)

def pair_features(a, b, projector):
    za, zb = projector.transform(np.stack([a,b]))
    delta=np.abs(za-zb)
    return np.array([float(a@b),float(normalize_vector(za)@normalize_vector(zb)),
                     delta.mean(),np.linalg.norm(delta),delta.max()],dtype=np.float32)

def fit_speaker_models(ids, cache, gold, normalizer):
    pair_records, link_x, link_y, link_weights, audit = [], [], [], [], []
    for sid in ids:
        pages = cache[sid]
        nodes, _ = cluster_nodes(pages)
        identity = {}
        pair_start = len(pair_records)
        for p, page in enumerate(pages):
            anchors = transcript_anchors(page, gold[sid][p], normalizer)
            votes = anchor_votes(page, anchors)
            for local, entries in votes.items():
                label = agreed_identity(entries)
                if label is not None:
                    identity[p, local] = label
            for t, speaker in anchors.items():
                candidates = []
                for c, local in enumerate(page['character_cluster_labels']):
                    # The current balloon must not manufacture its own training target.
                    label = agreed_identity(votes[int(local)], exclude=t)
                    if label is not None:
                        candidates.append((c, int(label == speaker)))
                if {y for _,y in candidates} != {0,1}:
                    continue
                counts = Counter(y for _,y in candidates)
                for c, target in candidates:
                    link_x.append(link_features(page,t,c));link_y.append(target)
                    link_weights.append(1/counts[target])
                    audit.append({'sequence_id':sid,'page':p+1,'text_index':t,
                                  'character_index':c,'same_speaker':target,
                                  'target_source':'other agreeing balloons in local cluster'})
        for i,a in enumerate(nodes):
            la=identity.get((a['page'],a['local']))
            if la is None:
                continue
            for b in nodes[:i]:
                lb=identity.get((b['page'],b['local']))
                if lb is not None and a['page'] != b['page']:
                    pair_records.append((a['embedding'],b['embedding'],int(la==lb),sid))
    if not pair_records or len({r[2] for r in pair_records}) < 2:
        raise RuntimeError('Not enough cross-page anchor pairs. Annotate development examples instead of inventing labels.')
    if len(set(link_y)) < 2:
        raise RuntimeError('Not enough independently anchored balloon candidates for ranker training.')
    embeddings=np.stack([e for a,b,_,_ in pair_records for e in (a,b)])
    projector=PCA(n_components=min(24,len(embeddings)-1,embeddings.shape[1]),random_state=42).fit(embeddings)
    px=np.stack([pair_features(a,b,projector) for a,b,_,_ in pair_records]);py=np.array([y for _,_,y,_ in pair_records])
    pair_counts=Counter(sid for *_,sid in pair_records)
    pw=np.array([1/pair_counts[sid] for *_,sid in pair_records]);pw*=len(pw)/pw.sum()
    def classifier(C):
        return make_pipeline(StandardScaler(),LogisticRegression(C=C,class_weight='balanced',max_iter=2000,random_state=42))
    matcher=classifier(.05).fit(px,py,logisticregression__sample_weight=pw)
    ranker=classifier(.1).fit(np.stack(link_x),link_y,logisticregression__sample_weight=link_weights)
    return {'projector':projector,'matcher':matcher,'ranker':ranker}, {
        'pair_counts':dict(Counter(map(int,py))), 'ranker_counts':dict(Counter(map(int,link_y))),
        'pair_sequences':len(pair_counts),'ranker_sequences':len({r['sequence_id'] for r in audit}),
        'ranker_training_balloons':len({(r['sequence_id'],r['page'],r['text_index']) for r in audit}),
        'supervision':'weak labels from agreeing OCR-aligned development balloons; no box-level ground truth'}, audit

def prepare_speaker_predictions(pages, models):
    nodes, appearances=cluster_nodes(pages)
    n=len(nodes); probabilities=np.eye(n);cosine=np.eye(n)
    pairs=[(i,j) for i,a in enumerate(nodes) for j,b in enumerate(nodes[:i]) if a['page']!=b['page']]
    if pairs:
        features=np.stack([pair_features(nodes[i]['embedding'],nodes[j]['embedding'],models['projector']) for i,j in pairs])
        scores=models['matcher'].predict_proba(features)[:,1]
        for (i,j),score in zip(pairs,scores):
            probabilities[i,j]=probabilities[j,i]=score
            cosine[i,j]=cosine[j,i]=float(nodes[i]['embedding']@nodes[j]['embedding'])
    ranking=[]
    for page in pages:
        nt,nc=len(page['ocr']),len(page['characters'])
        scores=np.zeros((nt,nc))
        if nt and nc:
            scores=models['ranker'].predict_proba(np.stack([
                link_features(page,t,c) for t in range(nt) for c in range(nc)]))[:,1].reshape(nt,nc)
        ranking.append(scores)
    return {'nodes':nodes,'appearances':appearances,'learned':probabilities,'cosine':cosine,'ranking':ranking}

def merge_prepared(prepared, learned, threshold):
    nodes=prepared['nodes'];matrix=prepared['learned' if learned else 'cosine']
    groups=[{i} for i in range(len(nodes))]
    edges=sorted([(matrix[i,j],i,j) for i,a in enumerate(nodes) for j,b in enumerate(nodes[:i]) if a['page']!=b['page']],reverse=True)
    for score,i,j in edges:
        if score<threshold: break
        a=next(g for g in groups if i in g);b=next(g for g in groups if j in g)
        if a is b or {nodes[k]['page'] for k in a}&{nodes[k]['page'] for k in b}: continue
        if min(matrix[x,y] for x in a for y in b)<threshold: continue
        groups=[g for g in groups if g is not a and g is not b]+[a|b]
    ids={n:i for i,g in enumerate(sorted(groups,key=min)) for n in g}
    return {key:ids[value] for key,value in prepared['appearances'].items()}

def upgraded_prediction(pages, prepared, cfg):
    identities=merge_prepared(prepared,cfg['learned_matcher'],cfg['match_threshold'])
    names={};output=[]
    for p,page in enumerate(pages):
        original={int(t):int(c) for t,c in page['text_character_associations']}
        rows=[]
        for t,raw in enumerate(page['ocr']):
            text=' '.join(str(raw).split())
            if not text or not page['is_essential_text'][t]: continue
            if page['narr_prob'][t]>=.85:
                speaker='NARRATION'
            else:
                chosen=original.get(t)
                if cfg['use_ranker'] and len(page['characters']):
                    scores=prepared['ranking'][p][t]
                    # Compare cluster scores, not two crops of the same character.
                    cluster_best={}
                    for c,local in enumerate(page['character_cluster_labels']):
                        if local not in cluster_best or scores[c]>cluster_best[local][0]:
                            cluster_best[local]=(scores[c],c)
                    ranked=sorted(cluster_best.values(),reverse=True)
                    best,candidate=ranked[0];runner=ranked[1][0] if len(ranked)>1 else 0.
                    if best>=cfg['link_threshold'] and best-runner>=cfg['link_margin']:
                        chosen=candidate
                key=('character',identities[p,chosen]) if chosen is not None else ('unresolved',p,t)
                if key not in names: names[key]=f'char{len(names)+1}'
                speaker=names[key]
            rows.append({'speaker':speaker,'text':text})
        output.append(rows)
    return output


## Anonymous identities across pages
Start with Magi's within-page character clusters. Merge clusters from different pages
only when every pair in the merged group passes the similarity threshold. Never merge
two different clusters from the same page. This conservative rule avoids chains of weak
matches, but cannot repair a bad within-page cluster.

Unassociated lines get separate unresolved IDs. That loses continuity for off-panel
speakers, but avoids pretending all off-panel speech comes from one person.

In [ ]:
def unit(x):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), 1e-8)

def character_ids(pages, threshold):
    nodes, appearance = [], {}
    for p, page in enumerate(pages):
        clusters = defaultdict(list)
        for c, label in enumerate(page['character_cluster_labels']):
            clusters[int(label)].append(c)
        for members in clusters.values():
            index = len(nodes)
            emb = unit(np.mean([unit(page['char_embeddings'][c]) for c in members], axis=0))
            nodes.append((p, emb))
            for c in members:
                appearance[p, c] = index
    groups = [{i} for i in range(len(nodes))]
    edges = sorted([(float(a[1] @ b[1]), i, j)
                    for i, a in enumerate(nodes) for j, b in enumerate(nodes[:i])
                    if a[0] != b[0]], reverse=True)
    for similarity, i, j in edges:
        if similarity < threshold:
            break
        gi = next(g for g in groups if i in g)
        gj = next(g for g in groups if j in g)
        if gi is gj:
            continue
        if {nodes[k][0] for k in gi} & {nodes[k][0] for k in gj}:
            continue
        if min(float(nodes[a][1] @ nodes[b][1]) for a in gi for b in gj) < threshold:
            continue
        merged = gi | gj
        groups = [g for g in groups if g is not gi and g is not gj] + [merged]
    mapping = {node: group for group, members in enumerate(sorted(groups, key=min)) for node in members}
    return {key: mapping[value] for key, value in appearance.items()}

def predict_pages(pages, heads=None, story_threshold=0.5, narration_threshold=0.7,
                  cluster_threshold=0.8):
    identities = character_ids(pages, cluster_threshold)
    names, output = {}, []
    for p, page in enumerate(pages):
        links = {int(t): int(c) for t, c in page['text_character_associations']}
        rows = []
        if heads and page['ocr']:
            features = np.stack([region_features(page, i) for i in range(len(page['ocr']))])
            keep_prob = heads['story'].predict_proba(features)[:, 1]
            narr_prob = heads['narration'].predict_proba(features)[:, 1]
        for i, raw in enumerate(page['ocr']):
            text = clean(raw)
            keep = keep_prob[i] >= story_threshold if heads and story_threshold is not None else page['is_essential_text'][i]
            if not text or not keep:
                continue
            if heads and narration_threshold is not None and narr_prob[i] >= narration_threshold:
                speaker = 'NARRATION'
            else:
                c = links.get(i)
                identity = ('character', identities[p, c]) if c is not None else ('unresolved', p, i)
                if identity not in names:
                    names[identity] = f'char{len(names)+1}'
                speaker = names[identity]
            rows.append({'speaker': speaker, 'text': text})
        output.append(rows)
    return output

def write_jsonl(path, predictions):
    with Path(path).open('w', encoding='utf-8') as f:
        for sid, pages in predictions.items():
            f.write(json.dumps({'sequence_id': sid, 'pages': pages}, ensure_ascii=False) + '\n')

def evaluate_config(name, config, active_heads):
    predictions = {sid: predict_pages(dev_cache[sid], active_heads, **config) for sid in val_ids}
    report = scorer.evaluate({sid: gold[sid] for sid in val_ids}, predictions)
    write_jsonl(WORK / f'{name}.jsonl', predictions)
    (WORK / f'{name}_scores.json').write_text(json.dumps(report, indent=2))
    return {'experiment': name, **config, **report['macro']}

## Compare the two additions
Fit on the 64 training sequences only. Compare the old pipeline, the matcher alone,
the ranker alone, and both together. This is 42 configurations. We choose the best
configuration with both components enabled; the table also shows whether that choice
beats the baseline. We do not hide a regression if the new models lose.

Text filtering, narration threshold and OCR are fixed here so that the experiment
isolates speaker changes. Reusing this validation set for earlier tuning makes these
results optimistic; a fresh grouped holdout is the next check.

In [ ]:
from functools import lru_cache
from rapidfuzz.distance import Levenshtein

speaker_models, speaker_counts, speaker_audit = fit_speaker_models(
    train_ids, dev_cache, gold, scorer.norm)
print(speaker_counts)
(WORK / 'speaker_training_counts.json').write_text(json.dumps(speaker_counts, indent=2))
pd.DataFrame(speaker_audit).to_csv(WORK / 'speaker_training_audit.csv', index=False)
joblib.dump(speaker_models, WORK / 'validation_speaker_models.joblib')

def add_narration_probabilities(pages, active_heads):
    for page in pages:
        if page['ocr']:
            features = np.stack([region_features(page, i) for i in range(len(page['ocr']))])
            page['narr_prob'] = active_heads['narration'].predict_proba(features)[:, 1]
        else:
            page['narr_prob'] = []
    return pages

prepared = {}
for sid in val_ids:
    add_narration_probabilities(dev_cache[sid], heads)
    prepared[sid] = prepare_speaker_predictions(dev_cache[sid], speaker_models)

# Only the implementation is accelerated; the selected output is officially rescored.
original_align = scorer.align_tokens
@lru_cache(maxsize=2048)
def alignment_indices(a, b):
    return original_align([(t, i) for i, t in enumerate(a)], [(t, i) for i, t in enumerate(b)])
def cached_align(a, b):
    return [(a[x[1]], b[y[1]], weight) for x, y, weight in alignment_indices(
        tuple(t for t, _ in a), tuple(t for t, _ in b))]
scorer.distance = Levenshtein.distance
scorer.align_tokens = cached_align

configs = [dict(learned_matcher=False, match_threshold=.7, use_ranker=False,
                link_threshold=.75, link_margin=.1)]
for learned in (False, True):
    for match in ((.35, .5, .65, .8, .9) if learned else (.7,)):
        if learned:
            configs.append(dict(learned_matcher=True, match_threshold=match,
                                use_ranker=False, link_threshold=.75, link_margin=.1))
        for threshold in (.55, .75, .9):
            for margin in (.1, .25):
                configs.append(dict(learned_matcher=learned, match_threshold=match,
                                    use_ranker=True, link_threshold=threshold, link_margin=margin))

refs = {sid: gold[sid] for sid in val_ids}
results = []
for i, cfg in enumerate(tqdm(configs, desc='Speaker experiments')):
    predictions = {sid: upgraded_prediction(dev_cache[sid], prepared[sid], cfg) for sid in val_ids}
    report = scorer.evaluate(refs, predictions)
    results.append({'experiment': f'speaker_{i:02d}', **cfg, **report['macro']})
comparison = pd.DataFrame(results)
comparison.to_csv(WORK / 'speaker_experiments.csv', index=False)
best = comparison[comparison.learned_matcher & comparison.use_ranker].sort_values(
    'balanced_joint_f1', ascending=False).iloc[0]
selected = configs[int(best.experiment.split('_')[1])]
(WORK / 'selected_speaker_config.json').write_text(json.dumps(selected, indent=2))
predictions = {sid: upgraded_prediction(dev_cache[sid], prepared[sid], selected) for sid in val_ids}
write_jsonl(WORK / f'{best.experiment}.jsonl', predictions)
official_report = official_score.evaluate(refs, official_score.load_jsonl(WORK / f'{best.experiment}.jsonl'))
for metric, value in official_report['macro'].items():
    assert np.isclose(value, float(best[metric]), atol=1e-10), metric
(WORK / f'{best.experiment}_scores.json').write_text(json.dumps(official_report, indent=2))
write_jsonl(WORK / 'validation_references.jsonl', refs)

ablation = []
for label, mask in [('Previous pipeline', ~comparison.learned_matcher & ~comparison.use_ranker),
                    ('Matcher only', comparison.learned_matcher & ~comparison.use_ranker),
                    ('Ranker only', ~comparison.learned_matcher & comparison.use_ranker),
                    ('Both', comparison.learned_matcher & comparison.use_ranker)]:
    row = comparison[mask].sort_values('balanced_joint_f1', ascending=False).iloc[0]
    ablation.append({'variant': label, 'text_order_score': row.text_order_score,
                     'balanced_joint_f1': row.balanced_joint_f1, 'joint_f1': row.joint_f1})
display(pd.DataFrame(ablation))
pd.DataFrame(ablation).to_csv(WORK / 'ablation_summary.csv', index=False)
if best.balanced_joint_f1 <= comparison.iloc[0].balanced_joint_f1:
    print('Both components did not improve the baseline. Keep the old submission and review training anchors.')
print('Selected settings:', selected)
print('Selected metrics agree with the unchanged official scorer.')
scorer.align_tokens = original_align
scorer.distance = official_score.distance


## Look at failures before the final run
The table shows the weakest validation sequences. Compare their images and transcripts;
separate detection misses, OCR errors, exclusions, ordering mistakes, and identity swaps.
Do not inspect or edit test predictions to tune the system.

In [ ]:
report = json.loads((WORK / f'{best.experiment}_scores.json').read_text())
worst = pd.DataFrame.from_dict(report['sequences'], orient='index')
display(worst[['text_order_score', 'balanced_joint_f1', 'matched_token_coverage']]
        .sort_values('balanced_joint_f1').head(8))
review_id = worst.sort_values('balanced_joint_f1').index[0]
predicted = scorer.load_jsonl(WORK / f'{best.experiment}.jsonl')[review_id]
for p in range(3):
    print(f'Page {p+1}')
    display(pd.DataFrame(predicted[p]))
    display(pd.DataFrame(gold[review_id][p]))
    display(Image.open(DATA_ROOT / seq_by_id[review_id]['images'][p]).resize((600, 850)))

## Refit and export
Refit the original small heads and the two speaker models on all 80 development
sequences. Keep the validation-selected settings fixed. Test images only supply
features; no test labels or manual edits are used. Save the complete fitted bundle
and the predictions together.

In [ ]:
final_heads, final_audit, final_counts = fit_heads(dev_ids)
final_speaker_models, final_speaker_counts, final_speaker_audit = fit_speaker_models(
    dev_ids, dev_cache, gold, scorer.norm)
bundle = {'heads': final_heads, 'speaker_models': final_speaker_models,
          'speaker_config': selected, 'metadata': metadata}
joblib.dump(bundle, WORK / 'trained_pipeline.joblib')
final_audit.to_csv(WORK / 'final_alignment_audit.csv', index=False)
pd.DataFrame(final_speaker_audit).to_csv(WORK / 'final_speaker_audit.csv', index=False)
(WORK / 'final_training_counts.json').write_text(json.dumps(final_counts, indent=2))
(WORK / 'final_speaker_training_counts.json').write_text(json.dumps(final_speaker_counts, indent=2))

def infer_test(bundle_path):
    fitted = joblib.load(bundle_path)
    assert fitted['metadata']['revision'] == revision
    predictions = {}
    for sid in tqdm(test_ids, desc='Test inference'):
        pages = add_narration_probabilities(extract_sequence(sid), fitted['heads'])
        prepared = prepare_speaker_predictions(pages, fitted['speaker_models'])
        predictions[sid] = upgraded_prediction(pages, prepared, fitted['speaker_config'])
    return predictions

test_predictions = infer_test(WORK / 'trained_pipeline.joblib')
submission = WORK / 'submission.jsonl'
write_jsonl(submission, test_predictions)
checked = official_score.load_jsonl(submission)
expected = official_score.load_jsonl(DATA_ROOT / 'sample_submission.jsonl')
assert set(checked) == set(expected) == set(test_ids)
assert len(checked) == 15
print('Validated:', submission)
print('Text entries:', sum(len(page) for pages in checked.values() for page in pages))


## Save the run
Download the archive and keep this executed notebook with it. The bundle includes the
fitted classifiers, PCA projection, settings, split, weak-label audits and comparisons.
The pretrained model and image-feature cache are not included in the ZIP.

The matcher and ranker are trained additions, but their anchors still depend on Magi's
links. Inspect development errors before drawing conclusions. Reading order, OCR and
story inclusion are unchanged. Neither model knows the true identity of an off-panel
speaker. The validation set has been reused, and no test accuracy is available.

Sources: [Magi](https://github.com/ragavsachdeva/magi),
[MagiV2](https://huggingface.co/ragavsachdeva/magiv2),
[Tails Tell Tales](https://arxiv.org/abs/2408.00298),
[scikit-learn](https://scikit-learn.org/stable/).
Write your own submission README, including the failed strict-anchor experiment and
the fact that a better balanced F1 does not mean every reported metric improved.

In [ ]:
import zipfile
archive = WORK.parent / 'manga_speaker_notebook_artifacts.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in WORK.iterdir():
        if path.is_file():
            z.write(path, path.name)
from IPython.display import FileLink
print('Saved archive:', archive)
print(f'Total elapsed since setup: {(time.perf_counter()-RUN_STARTED)/60:.1f} minutes')